# Dummy Variables — Solutions
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> ⚠️ **This file contains complete solutions. Release to students only after the submission deadline.**

In [ ]:
!pip install yfinance statsmodels --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
# Shared data — downloaded once
START, END = '2018-01-01', '2024-12-31'
px_ch = yf.download(['NESN.SW', '^SSMI'], start=START, end=END,
                    auto_adjust=True, progress=False)['Close']
ret_ch = px_ch.pct_change().dropna()
# rename by label, never by position: yfinance orders the Close columns alphabetically
ret_ch = ret_ch.rename(columns={'NESN.SW': 'NESN', '^SSMI': 'SMI'})[['NESN', 'SMI']]
ret_ch['D']   = (ret_ch.index >= '2020-03-16').astype(float)
ret_ch['D_x'] = ret_ch['D'] * ret_ch['SMI']

px_us = yf.download(['AAPL', '^GSPC'], start=START, end=END,
                    auto_adjust=True, progress=False)['Close']
ret_us = px_us.pct_change().dropna()
# rename by label, never by position: yfinance orders the Close columns alphabetically
ret_us = ret_us.rename(columns={'^GSPC': 'SP500'})[['AAPL', 'SP500']]
ret_us['D']   = (ret_us.index >= '2020-03-16').astype(float)
ret_us['D_x'] = ret_us['D'] * ret_us['SP500']
print(f'✓ Data loaded: {len(ret_ch)} Swiss days, {len(ret_us)} US days.')

---
# Solution 1 — Classify the Dummy and Count the Dummies

| # | Dummy type | # dummies | Reason |
|---|-----------|-----------|--------|
| a | **Event** | 1 | FOMC day vs. all other days — two groups → one dummy |
| b | **Regime** | 1 (+ 1 interaction) | before/after March 2022 — the *duration* question needs the slope term D·Δy |
| c | **Calendar (category)** | 3 | four quarters → 4 − 1 = 3 dummies, one quarter as reference |
| d | **Cross-sectional category** | 1 | payer vs. non-payer — two groups → one dummy |
| e | **Calendar** | 11 | twelve months → 11 dummies |
| f | **Regime** | 1 (+ 1 interaction for the slope) | pre/post SNB floor removal — two regimes → one dummy |

**Key pattern:** two groups always need exactly ONE dummy; m groups need m − 1. Whether you ALSO need interactions depends on whether the question is about a level or a slope.

---
# Solution 2 — Crisis Dummy for Nestlé (Intercept Dummy)

In [ ]:
X_lvl = sm.add_constant(ret_ch[['SMI', 'D']])
m_lvl = sm.OLS(ret_ch['NESN'], X_lvl).fit(cov_type='HC1')

print(f'β1_hat (SMI beta)    = {m_lvl.params["SMI"]:.4f}')
print(f'β2_hat (level shift) = {m_lvl.params["D"]:.6f}')
print(f'   t = {m_lvl.tvalues["D"]:.2f},  p = {m_lvl.pvalues["D"]:.3f}')

**Answers:**
1. $\hat{\beta}_2$ is the ceteris-paribus difference in Nestlé's average daily return between the post- and pre-COVID regime, holding the SMI return fixed — a pure level shift.
2. Typically NOT significant (|t| well below 1.96): Nestlé's average market-adjusted performance did not systematically shift after COVID.
3. Because $\beta_2$ only moves the line up or down. "Did the beta change?" is a question about the SLOPE — that needs the interaction term $D \cdot r_{SMI}$ (Exercise 3).

---
# Solution 3 — Nestlé's Beta Break (Slope Dummy)

In [ ]:
X_brk = sm.add_constant(ret_ch[['SMI', 'D', 'D_x']])
m_brk = sm.OLS(ret_ch['NESN'], X_brk).fit(cov_type='HC1')

b1, b3 = m_brk.params['SMI'], m_brk.params['D_x']
print(f'Pre-COVID beta:  β1_hat = {b1:.4f}')
print(f'Slope shift:     β3_hat = {b3:.4f}   (t = {m_brk.tvalues["D_x"]:.2f}, p = {m_brk.pvalues["D_x"]:.4f})')
print(f'Post-COVID beta: {b1 + b3:.4f}')
print(f'Relative change: {b3/b1*100:+.1f}%  of the pre-COVID beta')

**Answers:**
1. Read Nestlé's pre-COVID beta $\hat{\beta}_1$ off the table; the post-COVID beta is the sum $\hat{\beta}_1 + \hat{\beta}_3$. A defensive staple sits well below 1.
2. Decide from the $p$-value of $\hat{\beta}_3$ in your own output, then compare it with the $\hat{\beta}_3$ the lecture notebook produces for Apple. The lesson is that not every stock's risk profile changed with COVID — but which of the two is significant depends on the sample, so quote your numbers.
3. Worked example with round numbers: if $\hat{\beta}_3 = 0.08$ on a pre-period beta of 0.65, the absolute rise is 0.08 beta units and the relative rise is $0.08 / 0.65 \approx$ **+12.3%** — a relative change divides by the pre-period level, not by 1. Apply the same division to your own two numbers.

---
# Solution 4 — Joint Break Test, By Hand

In [ ]:
m_R = sm.OLS(ret_ch['NESN'], sm.add_constant(ret_ch['SMI'])).fit()
m_U = sm.OLS(ret_ch['NESN'], X_brk).fit()

R2_R, R2_U = m_R.rsquared, m_U.rsquared
q, df_ = 2, int(m_U.df_resid)
F_hand = ((R2_U - R2_R) / q) / ((1 - R2_U) / df_)
F_crit = stats.f.ppf(0.95, q, df_)
p_val  = 1 - stats.f.cdf(F_hand, q, df_)

print('Step 1.  H0: β2 = β3 = 0 (no break)  vs.  H1: at least one ≠ 0')
print(f'Step 2.  α = 5% → F_crit(2, {df_}) = {F_crit:.2f}')
print(f'Step 3.  R²_R = {R2_R:.4f}, R²_U = {R2_U:.4f} → F = {F_hand:.2f}  (p = {p_val:.4f})')
print(f'Step 4.  {"REJECT" if F_hand > F_crit else "DO NOT REJECT"} H0')
print('\nVerify:')
print(m_U.f_test('D = D_x = 0'))

**Answers:**
1. F and decision depend on the stock — for Nestlé the joint test is often borderline or insignificant, unlike Apple's clear rejection.
2. This is the **Chow test** for a structural break at a known date, in its dummy-variable form.
3. A break can show up in the level, the slope, or both. Testing only the individually significant term ignores the joint nature of "no break at all" — and with correlated regressors, individual t-tests and the joint F-test can disagree (as in the multicollinearity chapter).

---
# Solution 5 — The Trap, Demonstrated

In [ ]:
wd = pd.Series(ret_ch.index.dayofweek, index=ret_ch.index)
all_five = pd.get_dummies(wd, prefix='D').astype(float)
X_trap = sm.add_constant(all_five)

print(f'Columns: {X_trap.shape[1]},  rank: {np.linalg.matrix_rank(X_trap.values)}')
print(f'Row sums of dummy columns — min: {all_five.sum(axis=1).min()}, max: {all_five.sum(axis=1).max()}')

four = pd.get_dummies(wd, prefix='D', drop_first=True).astype(float)
X_ok = sm.add_constant(four)
print(f'\nAfter drop_first=True — columns: {X_ok.shape[1]},  rank: {np.linalg.matrix_rank(X_ok.values)}  ← full ✓')

**Answers:**
1. Rank 5 vs. 6 columns: one column is an exact linear combination of the others — $X^\top X$ is singular, OLS is not identified. Coefficients returned by a pseudo-inverse are arbitrary splits of the same fit.
2. Monday (`D_0`) was dropped and became the reference. The constant is the Monday mean; each remaining dummy coefficient is that day's average difference *to Monday*.
3. Alternative: keep all five dummies but **drop the intercept**. Less common because the coefficients then estimate absolute day means rather than differences — and the standard t-tests no longer answer the natural question "is this day different from the base?".

---
# Solution 6 — Day-of-Week Study on Nestlé

In [ ]:
y = ret_ch['NESN'] * 100
wd = pd.Series(ret_ch.index.dayofweek, index=ret_ch.index)
four = pd.get_dummies(wd, prefix='D', drop_first=True).astype(float)
four.columns = ['D_Tue', 'D_Wed', 'D_Thu', 'D_Fri']
m_dow = sm.OLS(y, sm.add_constant(four)).fit()

tab = pd.DataFrame({'estimate': m_dow.params, 'SE': m_dow.bse,
                    't': m_dow.tvalues, 'p': m_dow.pvalues}).round(4)
tab.index = ['α (Mon mean)', 'δ_Tue', 'δ_Wed', 'δ_Thu', 'δ_Fri']
print(tab)

means = [y[wd == d].mean() for d in range(5)]
ses   = [y[wd == d].std()/np.sqrt((wd == d).sum()) for d in range(5)]
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.bar(['Mon','Tue','Wed','Thu','Fri'], means,
       color=[ORANGE if m_ < 0 else YELLOW for m_ in means], edgecolor=GREY,
       yerr=[1.96*s for s in ses], capsize=6, error_kw=dict(ecolor=GREY, lw=1.2))
ax.axhline(0, color='black', lw=1)
ax.set_ylabel('Avg daily return (%)')
ax.set_title('NESN.SW — average return by weekday, 95% CIs', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

print(m_dow.f_test('D_Tue = D_Wed = D_Thu = D_Fri = 0'))

**Answers:**
1. Depends on the sample — differences are tiny fractions of a percent either way.
2. Typically no day is individually significant AND the joint F-test does not reject: no weekday pattern for Nestlé either.
3. Picking the best-looking day first and then t-testing it is data mining: the "best of five" is biased upward, and its t-statistic no longer has the standard distribution. The joint F-test asks the honest question with one controlled error rate.

---
# Solution 7 — January Effect

In [ ]:
px_sp = yf.download('^GSPC', start='2015-01-01', end='2024-12-31',
                    auto_adjust=True, progress=False)['Close'].squeeze()
r_sp = px_sp.pct_change().dropna() * 100
mo = pd.Series(r_sp.index.month, index=r_sp.index)
mdum = pd.get_dummies(mo, prefix='M', drop_first=True).astype(float)
m_jan = sm.OLS(r_sp, sm.add_constant(mdum)).fit()

print(f'α (January daily mean) = {m_jan.params["const"]:.4f}%')
print('\nMonthly δs (difference to January):')
print(m_jan.params.drop('const').round(4))
restr = ' = '.join(mdum.columns) + ' = 0'
print('\nJoint test (q = 11):')
print(m_jan.f_test(restr))

**Answers:**
1. If January were special, the δs would be predominantly negative (other months below January). In recent S&P 500 data the signs are mixed and small.
2. The joint F-test typically does not reject: no significant monthly seasonality in the broad index.
3. Once an anomaly is published, traders front-run it (buying in December kills the January premium) — the pattern is arbitraged away. Anomalies survive mainly where limits to arbitrage are strong (illiquid small caps, decades ago).

---
# Solution 8 — Turn-of-the-Month Dummy

In [ ]:
ym = r_sp.index.to_period('M')
rank_in_month = r_sp.groupby(ym).cumcount()
D_tom = (rank_in_month < 3).astype(float)
D_tom.index = r_sp.index

m_tom = sm.OLS(r_sp, sm.add_constant(pd.DataFrame({'D_TOM': D_tom}))).fit(cov_type='HC1')
print(f'α  (all other days mean)      = {m_tom.params["const"]:.4f}%')
print(f'δ  (turn-of-month premium)    = {m_tom.params["D_TOM"]:.4f}%')
print(f'   t = {m_tom.tvalues["D_TOM"]:.2f},  p = {m_tom.pvalues["D_TOM"]:.4f}')

**Answers:**
1. α is the average return on all NON-turn-of-month days (the reference group); δ is the extra average return on the first three trading days of a month.
2. Sample-dependent; in many recent windows the premium is positive but only borderline significant.
3. With a single dummy and no other regressor, the regression is identical to a **two-sample comparison of means** (turn-of-month days vs. the rest) — the t-test on δ is the two-group mean-difference test.

---
# Solution 9 — Equivalence Check

In [ ]:
X_us = sm.add_constant(ret_us[['SP500', 'D', 'D_x']])
m_full = sm.OLS(ret_us['AAPL'], X_us).fit()
b0, b1 = m_full.params['const'], m_full.params['SP500']
b2, b3 = m_full.params['D'],     m_full.params['D_x']

pre  = ret_us[ret_us['D'] == 0]
post = ret_us[ret_us['D'] == 1]
m_pre  = sm.OLS(pre['AAPL'],  sm.add_constant(pre['SP500'])).fit()
m_post = sm.OLS(post['AAPL'], sm.add_constant(post['SP500'])).fit()

print('              dummy regression      separate regressions')
print(f'pre  intercept   {b0: .6f}            {m_pre.params["const"]: .6f}')
print(f'pre  slope       {b1: .6f}            {m_pre.params["SP500"]: .6f}')
print(f'post intercept   {b0 + b2: .6f}            {m_post.params["const"]: .6f}')
print(f'post slope       {b1 + b3: .6f}            {m_post.params["SP500"]: .6f}')
ok = np.allclose([b0, b1, b0 + b2, b1 + b3],
                 [m_pre.params['const'], m_pre.params['SP500'],
                  m_post.params['const'], m_post.params['SP500']])
print(f'\nIdentical to machine precision? {ok} ✓')

**Answers:**
1. Yes — exactly. The fully interacted dummy model spans the same fit as two separate regressions: OLS minimises the same total sum of squared residuals, so the fitted lines coincide algebraically.
2. The dummy formulation delivers the comparison WITHIN one model — so you can directly TEST the differences (t on β₃, joint F on β₂, β₃) with correct standard errors.
3. The break test itself: two separate regressions give you two sets of coefficients but no immediate test statistic for "are they different?" — the dummy regression's β₂ and β₃ ARE those differences, ready to test. *(Small caveat: the pooled model imposes one common error variance across regimes; with HC-robust SEs that concern largely disappears.)*

---
# Solution 10 — Example: UBS and the Credit Suisse Takeover

In [ ]:
# Hypothesis: UBS's SMI beta changed after the CS takeover weekend (19 March 2023).
px = yf.download(['UBSG.SW', '^SSMI'], start='2021-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)['Close']
r = px.pct_change().dropna()
# rename by label, never by position: yfinance orders the Close columns alphabetically
r = r.rename(columns={'UBSG.SW': 'UBS', '^SSMI': 'SMI'})[['UBS', 'SMI']]
r['D']   = (r.index >= '2023-03-20').astype(float)
r['D_x'] = r['D'] * r['SMI']

m = sm.OLS(r['UBS'], sm.add_constant(r[['SMI', 'D', 'D_x']])).fit(cov_type='HC1')
b1, b3 = m.params['SMI'], m.params['D_x']
print(f'Pre-takeover beta:  {b1:.3f}')
print(f'Slope shift β3:     {b3:+.3f}  (t = {m.tvalues["D_x"]:.2f}, p = {m.pvalues["D_x"]:.4f})')
print(f'Post-takeover beta: {b1 + b3:.3f}')
print('\nJoint break test:')
print(m.f_test('D = D_x = 0'))

fig, ax = plt.subplots(figsize=(9, 5))
for grp, col, lab in [(r[r['D'] == 0], GREY, 'pre'), (r[r['D'] == 1], BLUE, 'post')]:
    ax.scatter(grp['SMI']*100, grp['UBS']*100, s=8, alpha=0.4, color=col, label=lab)
xx = np.linspace(r['SMI'].min(), r['SMI'].max(), 50)
ax.plot(xx*100, (m.params['const'] + b1*xx)*100, color='black', lw=2, label=f'pre slope {b1:.2f}')
ax.plot(xx*100, (m.params['const'] + m.params['D'] + (b1+b3)*xx)*100, color=RED, lw=2,
        label=f'post slope {b1+b3:.2f}')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=4, frameon=False, fontsize=10)
ax.set_xlabel('SMI daily return (%)'); ax.set_ylabel('UBS daily return (%)')
ax.set_title('UBS beta before/after the CS takeover', fontweight='bold', loc='left', pad=30)
plt.tight_layout(); plt.show()

**Executive summary (example):**
UBS's market beta was elevated already before the takeover (a cyclical bank), and the point estimate rises further after March 2023 as the balance sheet doubled and integration risk entered the price. Whether the slope shift is statistically significant depends on the window — the takeover coincided with a volatile banking-sector episode, which widens the standard errors. The joint Chow test often rejects "no break" mainly through the level term in the weeks around the event. Caveat: a single known break date chosen *because* we saw the event in the news is mild look-ahead — the sup-F scan in the challenge is the honest version.

---
# 🔥 Challenge Solution — Mini sup-F Scan

In [ ]:
y_us = ret_us['AAPL']
n = len(ret_us)
lo, hi = int(0.10 * n), int(0.90 * n)
candidates = range(lo, hi, 21)          # every ~month

Fs, dates = [], []
for i in candidates:
    Dv = (np.arange(n) >= i).astype(float)
    Xc = sm.add_constant(pd.DataFrame({
        'SP500': ret_us['SP500'].values,
        'D': Dv,
        'D_x': Dv * ret_us['SP500'].values}, index=ret_us.index))
    mc_ = sm.OLS(y_us, Xc).fit()
    Fs.append(float(np.squeeze(mc_.f_test('D = D_x = 0').fvalue)))
    dates.append(ret_us.index[i])

Fs = np.array(Fs)
best = np.argmax(Fs)
print(f'Maximum F = {Fs[best]:.2f} at candidate break date {dates[best].date()}')

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(dates, Fs, color=RED, lw=1.6)
ax.axhline(3.00, color=GREY, ls='--', lw=1, label='3.00 (single-date 5% critical value)')
ax.axvline(dates[best], color=BLUE, ls=':', lw=1.4, label=f'max F: {dates[best].date()}')
ax.set_ylabel('Joint F-statistic (q = 2)')
ax.set_title('Mini sup-F scan — Apple/S&P 500 break search', fontweight='bold', loc='left')
ax.legend(loc='upper right', frameon=False)
plt.tight_layout(); plt.show()

print('\nThe scan typically peaks in spring 2020 — the data locate the COVID break on their own.')
print('Caveat: the 3.00 line is NOT the right critical value for a SEARCHED maximum —')
print('the sup-F (Quandt-Andrews) distribution has higher critical values, because scanning')
print('many dates inflates the best F under the null.')

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*⚠️ Release to students only after the submission deadline.*